# Sketchy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sketchy.sketchy_dataset import SketchyDataset

In [ ]:
dataset_root = "<path/to/sketchy/root>"
split = "train"
img_size = 512
load_img = True
load_full_sketch = True
load_partial_sketches = True
with_shoes = False
concat_partials = True
compose_full_sketch = True
img_transforms = None
mask_transforms = None
full_sketch_transforms = None


sketchy_dataset = SketchyDataset(dataset_root=dataset_root, 
                              split=split, 
                              img_size=img_size, 
                              load_img=load_img, 
                              load_full_sketch=load_full_sketch,
                              load_partial_sketch=load_partial_sketches,
                              img_transforms=img_transforms, 
                              mask_transforms=mask_transforms,
                              full_sketch_transforms=full_sketch_transforms,
                              with_shoes=with_shoes,
                              concat_partials=concat_partials,
                              compose_full_sketch=compose_full_sketch,
                              )
print(f"Number of images in {split} split: {len(sketchy_dataset)}")

In [ ]:
RESHAPE_SIZE = img_size
idx = 100  # index of the image to show
batch = sketchy_dataset.__getitem__(idx)
img_data = batch["img_data"]
annotations = batch["annotations"]
img_path = batch['image_path']
img_id = batch['image_id']
total_mask = np.zeros((RESHAPE_SIZE, RESHAPE_SIZE), dtype=np.uint8)
partial_descs = []
partial_masks = []
partial_sketches = []
categories = []
for ann in annotations:
    mask = sketchy_dataset.ann2Mask(ann)
    partial_masks.append(mask)
    total_mask += mask
    partial_descs += [ann["description"]]
    categories += [ann["category_name"]]
# set all values >0 to 1
total_mask[total_mask > 0] = 1
# print the description
print(f"{img_id}: {batch['description']}")

fig, axs = plt.subplots(1, 3, figsize=(10, 5))
img = Image.open(img_path).resize((RESHAPE_SIZE, RESHAPE_SIZE))
axs[0].imshow(img)
axs[0].axis("off")
axs[0].set_title("Original image")
axs[1].imshow(total_mask, cmap="gray")
axs[1].axis("off")
axs[1].set_title("Mask")
axs[2].imshow(img)
axs[2].imshow(total_mask, alpha=0.5, cmap="viridis")
axs[2].axis("off")
axs[2].set_title("Overlay")
fig.tight_layout()
fig.show()


# plot partial masks and their descriptions, allow for single or multiple partial masks
if len(partial_masks) == 1:
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(partial_masks[0], cmap="gray")
    axs[0].axis("off")
    axs[0].set_title(f"{categories[0]}: {partial_descs[0]}")
    axs[1].imshow(batch['partial_sketches'][0])
    axs[1].axis("off")
else:
    fig, axs = plt.subplots(len(partial_masks), 2, figsize=(10, 5))
    for i, mask in enumerate(partial_masks):
        axs[i,0].imshow(mask, cmap="gray")
        axs[i,0].axis("off")
        axs[i,0].set_title(f"{categories[i]}: {partial_descs[i]}")
        axs[i,1].imshow(batch['partial_sketches'][i])
        axs[i,1].axis("off")

In [ ]:
TO_SHOW = 5
RESHAPE_SIZE = img_size
start_idx = 100
end_idx = start_idx + TO_SHOW
item_idx = start_idx
while item_idx < end_idx:
    idx = item_idx
    batch = sketchy_dataset.__getitem__(idx)
    img_data = batch["img_data"]
    annotations = batch["annotations"]
    img_path = batch['image_path']
    img_id = batch['image_id']
    partial_descs = batch['partial_descriptions']
    partial_sketches = batch['partial_sketches']
    sketches_paths = batch['partial_sketches_paths']

    print(f"{idx} : {img_id}")
    for pdesc in partial_descs:
        print(f"{pdesc}")

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    img = Image.open(img_path).resize((RESHAPE_SIZE, RESHAPE_SIZE))
    ax.imshow(img)
    ax.axis("off")

    # plot partial masks and their descriptions, allow for single or multiple partial masks
    if len(partial_sketches) == 1:
        fig, axs = plt.subplots(1, 1, figsize=(10, 5))
        axs.imshow(partial_sketches[0], cmap="gray")
        axs.axis("off")

    else:
        fig, axs = plt.subplots(1, len(partial_sketches), figsize=(10, 5))
        for i, mask in enumerate(partial_sketches):
            axs[i].imshow(partial_sketches[i])
            axs[i].axis("off")
    # fig.show()
    plt.show()
    item_idx += 1